In [9]:
import polars as pl
from pathlib import Path


data_dir = Path("../../data/MergeData")



In [10]:
#  data set: Weather data hourly
df_weather = pl.read_csv(data_dir / "weather_hourly.csv").with_columns(
    pl.col("date").str.to_datetime("%d.%m.%Y %H:%M")
).rename({"location_Zst": "Zst"})

print("finished")

finished


In [11]:
#  data set: BAst data

df_BASt = pl.read_csv(data_dir / "kiel_specific_stations_raw.csv").with_columns(
    (
        pl.col("Datum").cast(pl.Utf8).str.to_datetime("%y%m%d") + 
        
        pl.duration(hours=pl.col("Stunde").cast(pl.Int64) - 1)
    )
    .alias("date")
).drop(["Datum", "Stunde"])
print("finished")

finished


In [12]:
# Datei 3
df_air = pl.read_csv(data_dir / "air_quality_hourly.csv").with_columns(
    pl.col("date").str.to_datetime("%d.%m.%Y %H:%M")
).rename({"location_Zst": "Zst"})

print("finished")

finished


In [13]:


# Join air and weather
df_combined = df_air.join(
    df_weather, 
    on=["date", "Zst"], 
    how="inner"
)

print("finished")

finished


In [14]:
# Join BASt
df_final = df_combined.join(
    df_BASt, 
    on=["date", "Zst"], 
    how="full", 
    suffix="_3"
)

print("finished")

finished


In [15]:
# final touch
df_final = df_final.sort(["date", "Zst"])
#df_final = df_combined

df_final = df_final.with_columns(
    pl.col("date").dt.strftime("%d.%m.%Y %H:%M")
)

df_finalclear = df_final.drop(["date_3", "Zst_3", "TKNR", "location_name_right", "Fahrtzw", "ozone", "sulphur_dioxide","name"])

df_finalclear.write_csv(data_dir / "holy_file.csv")

print("finished")

finished


In [16]:
print(f"Air: {df_air.height} Zeilen")
print(f"Weather: {df_weather.height} Zeilen")
print(f"BASt: {df_BASt.height} Zeilen")

print(f"Combined: {df_combined.height} Zeilen")

Air: 350592 Zeilen
Weather: 350592 Zeilen
BASt: 297288 Zeilen
Combined: 350592 Zeilen


In [ ]:
# 